In [1]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import zmq

from bcs import BCSz


In [2]:
server = BCSz.BCSServer()

In [3]:
conn = await server.connect(addr="localhost", port=5577)

Server Public Key b'3=KZYI@pOlgBr1svn7=c7$Xh+f>U^t9S58jP]h&a'


In [4]:
await server.start_instrument_acquire("Axis Photonique", acq_time_s=.002)

{'success': True,
 'error description': 'no error',
 'log?': True,
 'elapsed_s': 0.560043500001484,
 'API_delta_t': 0.570124626159668}

In [5]:
start = perf_counter()

img = await server.get_instrument_acquired2d_string("Axis Photonique")
final_acq = perf_counter() - start

print(final_acq)

height, width, s = img["Height"], img["Width"], img["Data"]
# extract the array from the string
parts = [x for x in s.strip().split(",") if x]
array = np.array(parts, dtype=np.int32).reshape(height, width)


final_acq = perf_counter() - start

print(final_acq)

6.137620899826288
11.462455799803138


In [6]:
start = perf_counter()
img = await server.get_instrument_acquired2d_base85("Axis Photonique")
final_acq = perf_counter() - start

print(final_acq)

height, width, s = img["Height"], img["Width"], img["Data"]

1.236607600003481


In [7]:
t0 = perf_counter()
raw = zmq.utils.z85.decode(s) # no need to encode the string first this saves a second
t1 = perf_counter()
arr = np.frombuffer(raw, dtype=np.uint8)
t2 = perf_counter()
img = arr.reshape(height, width)
t3 = perf_counter()


In [8]:
print(f"z85 decode: {t1 - t0: .2g}s")
print(f"frombuffer: {t2 - t1: .2g}s")
print(f"reshape: {t3 - t2: .2g}s")

z85 decode:  6.8s
frombuffer:  0.00015s
reshape:  0.00011s


In [9]:
_Z85_ALPHABET = b'0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ.-:+=^!/*?&<>()[]{}@%$#'
_Z85_DEC = np.full(255, 0, np.uint64)
for i, c in enumerate(_Z85_ALPHABET):
    _Z85_DEC[c] = i
_Z85_POW = np.array([85**i for i in range(4, -1, -1)], dtype=np.uint64)

def z85decode_vectorized(data: str | bytes) -> np.ndarray:
    b=np.frombuffer(data if isinstance(data, bytes) else data.encode(), dtype=np.uint8)
    digits = _Z85_DEC[b].reshape(-1, 5)
    values = (digits * _Z85_POW).sum(axis=1).astype(np.uint32)
    return values.astype(">u4").view(np.uint8)

In [10]:
t0_2 = perf_counter()
raw = z85decode_vectorized(s)
t1_2 = perf_counter()
arr = np.frombuffer(raw, dtype=np.uint8)
t2_2 = perf_counter()
img = arr.reshape(height, width)
t3_2 = perf_counter()

In [11]:
print(f"z85_vectorized: {t1_2 - t0_2: .2g}s")
print(f"frombuffer: {t2_2 - t1_2: .2g}s")
print(f"reshape: {t3_2 - t2_2: .2g}s")

z85_vectorized:  0.43s
frombuffer:  0.00019s
reshape:  0.0021s


In [15]:
# a little bit of rust magic 
from bcs_rs import decode_z85_parallel, decode_z85 

In [16]:
t0_3 = perf_counter()
raw = decode_z85(s)
t1_3 = perf_counter()
t0_4 = perf_counter()
raw = decode_z85_parallel(s)
t1_4 = perf_counter()

In [17]:
print(f"decode_z85: {t1_3 - t0_3: .2g}s")
print(f"decode_z85_parallel: {t1_4 - t0_4: .2g}s")

decode_z85:  0.062s
decode_z85_parallel:  0.035s


In [ ]:
array = np.frombuffer(zmq.utils.z85.decode(s), dtype=np.uint8).reshape(height, width)

acq_zmq = perf_counter() - start

print(acq_zmq)

array_2 = _parse_payload(img)

In [ ]:
api_call_start = perf_counter()
await server._zmq_socket.send(...)
t_after_send = perf_counter()
response_dict = json.loads(await self._zmq_socket.recv())
t_after_recv = perf_counter()
print(f"send: {(t_after_send - api_call_start)*1000:.0f} ms, recv: {(t_after_recv - t_after_send)*1000:.0f} ms")

In [ ]:

plt.imshow(array)
plt.show()

final_acq = perf_counter() - start

print(final_acq)